# **Regression: Ridge Regression (L2 Regularization)**

## **Justification of Preprocessing Strategy**

### **The Mathematical Necessity of Scaling for L2 Penalty**
**Ridge Regression** introduces an L2 regularization penalty (also known as Tikhonov regularization) to the Ordinary Least Squares (OLS) loss function. This penalty adds the squared magnitude of the coefficients as a constraint to the optimization objective.


Because the penalty squares the $\beta$ weights, any feature with a large numerical scale will naturally force its corresponding coefficient to be extremely small to avoid a massive penalty explosion. This penalization occurs regardless of the feature's actual clinical relevance to the `diabetes_risk_score`. To prevent scale-driven bias, the feature space must be uniform. We will evaluate both **Standardization (StandardScaler)** and **Normalization (MinMaxScaler)** across all optimization levels to determine which data representation minimizes prediction errors.

### **Combating Multicollinearity without Sparsity**
Unlike Lasso (L1), which drives coefficients exactly to zero to perform feature selection, Ridge (L2) shrinks coefficients **proportional to their variance**, bringing them close to zero but never fully eliminating them. This makes Ridge highly effective when dealing with multicollinearity arising from numerous dummy-encoded categorical variables (e.g., ethnicity, employment status), as it retains all predictors while safely distributing the weight among correlated features. To protect model validity, classification targets are removed, and **stratification is omitted** due to the continuous nature of the target.



## **Experiment Design**

We have established a comprehensive tournament consisting of **6 distinct runs** crossing both scalers with 3 levels of hyperparameter exploration to optimize **MAE, RMSE, and $R^2$**:

* **Run 1 & 2: Ridge Baseline** — Running the Ridge model with strict Scikit-Learn default parameters (`alpha=1.0`) under **Standardization** vs. **Normalization**.
* **Run 3 & 4: GridSearchCV Tuning** — Conducting an exhaustive grid search over a discrete set of fixed `alpha` values under **Standardization** vs. **Normalization**.
* **Run 5 & 6: Optuna Optimization** — Leveraging Bayesian optimization to continuously explore a fine-grained logarithmic range of the L2 `alpha` penalty under **Standardization** vs. **Normalization**.

For both GridSearchCV and Optuna runs, internal evaluation is strictly validated using **3-Fold Cross-Validation**.



In [2]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Ridge")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Drop classification targets to avoid data leakage
X = df_final.drop(["diabetes_risk_score", "diagnosed_diabetes", "diabetes_stage"], axis=1, errors='ignore')
y = df_final['diabetes_risk_score']

# Split data (80/20) - Continuous target means NO stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_regression_metrics(y_true, y_pred, duration):
    """Utility function to log regression metrics to MLflow"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("fit_time", duration)

# Define the scaling strategies to compare across the entire tournament
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

# ---------------------------------------------------------
# RUN 1 & 2: RIDGE BASELINE (Strict Defaults)
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Ridge_Baseline_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        # OLS with default alpha=1.0
        model = Ridge(random_state=42)
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        mlflow.log_param("optimization", "none_default")
        mlflow.log_param("scaler", s_name)
        mlflow.log_params(model.get_params())
        
        log_regression_metrics(y_test, model.predict(X_test_scaled), duration)

# ---------------------------------------------------------
# RUN 3 & 4: GRIDSEARCHCV TUNING
# ---------------------------------------------------------
param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 500.0]}

for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Ridge_GridSearch_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        grid = GridSearchCV(
            Ridge(random_state=42), 
            param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        best_model = grid.best_estimator_
        
        mlflow.log_param("optimization", "GridSearchCV")
        mlflow.log_param("scaler", s_name)
        mlflow.log_params(grid.best_params_)
        
        log_regression_metrics(y_test, best_model.predict(X_test_scaled), duration)

# ---------------------------------------------------------
# RUN 5 & 6: OPTUNA BAYESIAN OPTIMIZATION
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
    
    def objective(trial):
        # Continuous exploration of alpha using log scale for fine-tuning
        alpha_val = trial.suggest_float("alpha", 1e-3, 1000.0, log=True)
        solver_val = trial.suggest_categorical("solver", ["auto", "saga"])
        
        model = Ridge(alpha=alpha_val, solver=solver_val, random_state=42)
        
        # Target: Minimize MAE (negate negative MAE back to positive)
        scores = -cross_val_score(
            model, X_train_scaled, y_train, 
            cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        return scores.mean()

    with mlflow.start_run(run_name=f"Ridge_Optuna_{s_name}"):
        study = optuna.create_study(direction="minimize")
        start_time = time.time()
        study.optimize(objective, n_trials=15)
        duration = time.time() - start_time
        
        best_ridge_opt = Ridge(**study.best_params, random_state=42)
        best_ridge_opt.fit(X_train_scaled, y_train)
        
        mlflow.log_param("optimization", "optuna")
        mlflow.log_param("scaler", s_name)
        mlflow.log_params(study.best_params)
        
        log_regression_metrics(y_test, best_ridge_opt.predict(X_test_scaled), duration)

[I 2026-05-20 10:13:24,375] A new study created in memory with name: no-name-3b12ff32-0083-4c25-b656-12b02081d942
[I 2026-05-20 10:13:24,759] Trial 0 finished with value: 0.40122749464013424 and parameters: {'alpha': 121.33518974982987, 'solver': 'auto'}. Best is trial 0 with value: 0.40122749464013424.
[I 2026-05-20 10:13:25,917] Trial 1 finished with value: 0.3985210316246679 and parameters: {'alpha': 48.466053609863344, 'solver': 'saga'}. Best is trial 1 with value: 0.3985210316246679.
[I 2026-05-20 10:13:27,583] Trial 2 finished with value: 0.397041957352422 and parameters: {'alpha': 0.004747921409771724, 'solver': 'saga'}. Best is trial 2 with value: 0.397041957352422.
[I 2026-05-20 10:13:28,056] Trial 3 finished with value: 0.39704400131180967 and parameters: {'alpha': 0.06466365685755952, 'solver': 'auto'}. Best is trial 2 with value: 0.397041957352422.
[I 2026-05-20 10:13:28,423] Trial 4 finished with value: 0.39704281345637665 and parameters: {'alpha': 0.013407053040359678, 's

## **Winner Run Selection (Priority Elimination Framework)**

### **Selection Criteria (in priority order)**
1. **Priority 1 (60% weight): Lowest MAE** — Clinical proximity; minimizes average day-to-day prediction error.
2. **Priority 2 (30% weight): RMSE proportional to MAE** — Rejects runs where RMSE spikes relative to MAE, indicating catastrophic errors.
3. **Priority 3 (10% weight): Acceptable R²** — Confirms statistical fit quality.
4. **Tiebreaker: Lowest Fit Time** — Applied only if a technical tie exists in MAE, RMSE, and R².

### **All Runs: Summary Table with Metrics**

| Run | Optimization | Scaler | Alpha | MAE | RMSE | R² Score | Fit Time (s) | 
|---|---|---|---:|---:|---:|---:|---:|
| **Ridge_GridSearch_Standardization** | **GridSearchCV** | **Standardization** | **0.01** | **0.40386** | **0.71129** | **0.993869** | **6.50** |
| Ridge_Optuna_Standardization | Optuna | Standardization | 0.001299 | 0.40386 | 0.71129 | 0.993869 | 18.38 | 
| Ridge_Optuna_Normalization | Optuna | Normalization | 0.001026 | 0.40386 | 0.71129 | 0.993869 | 32.06 | 
| Ridge_GridSearch_Normalization | GridSearchCV | Normalization | 0.01 | 0.40387 | 0.71129 | 0.993869 | 1.72 | 
| Ridge_Baseline_Standardization | None (Default) | Standardization | 1.0 | 0.40388 | 0.71129 | 0.993869 | 0.07 |
| Ridge_Baseline_Normalization | None (Default) | Normalization | 1.0 | 0.40550 | 0.71138 | 0.993868 | 0.06 | 

### **Step-by-Step Elimination Process**

**Step 1: Filter by Lowest MAE (Priority 1 — 60%)**
- Threshold: MAE ≤ 0.40386
- Candidates passing: Ridge_GridSearch_Standardization (0.40386), Ridge_Optuna_Standardization (0.40386), Ridge_Optuna_Normalization (0.40386)
- Eliminated: Ridge_GridSearch_Normalization, both baselines

**Step 2: Verify RMSE Proportional to MAE (Priority 2 — 30%)**
- All three candidates: RMSE = 0.71129 (identical)
- MAE-to-RMSE ratio: 0.40386 / 0.71129 ≈ 0.568 (same for all)
- Status: **No catastrophic divergence detected**. All three candidates pass.

**Step 3: Confirm Acceptable R² (Priority 3 — 10%)**
- All three candidates: R² = 0.993869 (identical, excellent fit)
- Status: All three candidates confirmed acceptable.

**Step 4: Apply Tiebreaker — Lowest Fit Time**
- Ridge_GridSearch_Standardization: 6.50 s ← **LOWER**
- Ridge_Optuna_Standardization: 18.38 s
- Ridge_Optuna_Normalization: 32.06 s

### **Final Decision**
**Winner: Ridge_GridSearch_Standardization**

**Justification:** Three runs achieve identical MAE, RMSE, and R² performance. GridSearch_Standardization wins via the fit-time tiebreaker (6.50 s), being significantly faster than Optuna_Standardization (18.38 s) and Optuna_Normalization (32.06 s), while maintaining identical predictive quality. This makes it the most efficient choice for production deployment.